# 🐧 Linux Container Runtime
### Using Namespaces, Seccomp and Cgroups

> **Requirements:** Linux (Kali / Ubuntu), root access, Jupyter running natively on Linux

---
## Steps
1. Install dependencies
2. Write the C source code
3. Compile the program
4. Verify the binary
5. Run the container
6. Verify isolation (inside container)

## Step 1 — Install Dependencies

In [ ]:
import subprocess

result = subprocess.run(
    ['sudo', 'apt-get', 'install', '-y', 'gcc', 'libseccomp-dev'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else '')
print(result.stderr[-500:]  if result.stderr else '')
print('✅ Done.' if result.returncode == 0 else '❌ Failed.')

## Step 2 — Write the C Source Code

In [ ]:
c_code = r"""
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sched.h>
#include <unistd.h>
#include <sys/mount.h>
#include <sys/wait.h>
#include <sys/types.h>
#include <sys/stat.h>
#include <fcntl.h>
#include <seccomp.h>
#include <errno.h>

#define STACK_SIZE      (1024 * 1024)
#define CGROUP_PATH     "/sys/fs/cgroup/mycontainer"
#define HOSTNAME        "mycontainer"
#define MEM_LIMIT       "268435456"
#define CPU_LIMIT       "50000 100000"

static char child_stack[STACK_SIZE];

static int write_file(const char *path, const char *value) {
    FILE *f = fopen(path, "w");
    if (!f) {
        fprintf(stderr, "[-] Cannot open %s: %s\n", path, strerror(errno));
        return -1;
    }
    fprintf(f, "%s\n", value);
    fclose(f);
    return 0;
}

static void setup_seccomp(void) {
    printf("[*] Applying seccomp filters...\n");
    scmp_filter_ctx ctx = seccomp_init(SCMP_ACT_ALLOW);
    if (!ctx) { perror("seccomp_init"); exit(1); }

    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(ptrace),     0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(reboot),     0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(kexec_load), 0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(mount),      0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(umount2),    0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(pivot_root), 0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(swapon),     0);
    seccomp_rule_add(ctx, SCMP_ACT_KILL, SCMP_SYS(swapoff),    0);

    if (seccomp_load(ctx) < 0) { perror("seccomp_load"); exit(1); }
    seccomp_release(ctx);
    printf("[+] Seccomp filters applied.\n");
}

static void setup_cgroups(void) {
    printf("[*] Setting up cgroups...\n");
    if (mkdir(CGROUP_PATH, 0755) < 0 && errno != EEXIST) {
        fprintf(stderr, "[-] mkdir cgroup: %s\n", strerror(errno));
        exit(1);
    }
    char mem_path[256], cpu_path[256], procs_path[256];
    snprintf(mem_path,   sizeof(mem_path),   "%s/memory.max",   CGROUP_PATH);
    snprintf(cpu_path,   sizeof(cpu_path),   "%s/cpu.max",      CGROUP_PATH);
    snprintf(procs_path, sizeof(procs_path), "%s/cgroup.procs", CGROUP_PATH);
    write_file(mem_path, MEM_LIMIT);
    write_file(cpu_path, CPU_LIMIT);
    char pid_str[32];
    snprintf(pid_str, sizeof(pid_str), "%d", getpid());
    write_file(procs_path, pid_str);
    printf("[+] Cgroups configured.\n");
}

static int child_func(void *arg) {
    (void)arg;
    printf("[*] Setting hostname to '%s'...\n", HOSTNAME);
    if (sethostname(HOSTNAME, strlen(HOSTNAME)) < 0) { perror("sethostname"); return 1; }
    printf("[+] Hostname set.\n");

    printf("[*] Configuring mount namespace...\n");
    if (mount(NULL, "/", NULL, MS_REC | MS_PRIVATE, NULL) < 0) { perror("mount private"); return 1; }
    if (mount("proc", "/proc", "proc", 0, NULL) < 0) { perror("mount proc"); return 1; }
    printf("[+] Filesystem isolation done.\n");

    setup_seccomp();

    printf("\n[+] Container ready! Launching shell...\n");
    printf("    Hostname : %s\n", HOSTNAME);
    printf("    PID      : %d\n\n", getpid());

    char *args[] = {"/bin/bash", NULL};
    execv("/bin/bash", args);
    perror("execv");
    return 1;
}

int main(void) {
    if (getuid() != 0) {
        fprintf(stderr, "[-] Must run as root.\n");
        exit(1);
    }
    printf("=== Linux Container Runtime ===\n\n");
    setup_cgroups();
    printf("[*] Cloning child process with new namespaces...\n");
    pid_t pid = clone(child_func, child_stack + STACK_SIZE,
                      CLONE_NEWPID | CLONE_NEWNS | CLONE_NEWUTS | SIGCHLD, NULL);
    if (pid < 0) { perror("clone"); exit(1); }
    printf("[+] Container started with host PID: %d\n\n", pid);
    waitpid(pid, NULL, 0);
    printf("\n[+] Container exited cleanly.\n");
    return 0;
}
"""

with open('container.c', 'w') as f:
    f.write(c_code)

print('✅ container.c written successfully!')
print(f'   Lines of code: {len(c_code.splitlines())}')

## Step 3 — Compile the C Code

In [ ]:
result = subprocess.run(
    ['gcc', '-Wall', '-O2', '-o', 'container', 'container.c', '-lseccomp'],
    capture_output=True, text=True
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)

if result.returncode == 0:
    print('✅ Compilation successful!')
else:
    print('❌ Compilation failed. Check STDERR above.')

## Step 4 — Verify the Binary

In [ ]:
import os

binary = './container'
if os.path.exists(binary):
    size = os.path.getsize(binary)
    print(f'✅ Binary exists: {binary}  ({size} bytes)')
    
    # Show file info
    r = subprocess.run(['file', binary], capture_output=True, text=True)
    print(r.stdout)
else:
    print('❌ Binary not found. Run Step 3 first.')

## Step 5 — Run the Container

> ⚠️ **Note:** An interactive shell cannot run inside Jupyter. Run this in your Linux terminal:
>
> ```bash
> sudo ./container
> ```
>
> The cell below performs a **non-interactive test** to verify the binary starts correctly.

In [ ]:
# Non-interactive test: run container and immediately send 'exit'
result = subprocess.run(
    ['sudo', './container'],
    input='exit\n',
    capture_output=True,
    text=True,
    timeout=10
)

print('=== OUTPUT ===')
print(result.stdout)
if result.stderr:
    print('=== STDERR ===')
    print(result.stderr)

if result.returncode == 0:
    print('\n✅ Container ran and exited successfully!')
else:
    print(f'\n⚠️  Exit code: {result.returncode}')

## Step 6 — Verify Isolation (Run These Inside the Container Shell)

After running `sudo ./container` in terminal, execute these commands **inside** the container:

```bash
# 1. PID Namespace — should print 1
echo $$

# 2. Hostname Isolation — should print: mycontainer
hostname

# 3. Process Visibility — only container processes visible
ps aux

# 4. Cgroup membership
cat /proc/self/cgroup

# 5. Test seccomp block (ptrace should be killed)
# strace ls   # this should be blocked by seccomp

# Exit container
exit
```

## Summary

| Feature | Mechanism | Status |
|---|---|---|
| Process Isolation | PID Namespace | ✅ |
| Filesystem Isolation | Mount Namespace | ✅ |
| Hostname Isolation | UTS Namespace | ✅ |
| Syscall Filtering | Seccomp | ✅ |
| Resource Limiting | Cgroups v2 | ✅ |
